# Gold Price 💰

In this notebook, predictive analytics study was performed on a given data set which consists of the troy ounce gold prices over time horizon.

The breakdown of the study is as below:

<ul>
<li>Raw data extraction</li>
<li>Data transformation</li>
<li>Data visualization</li>
<li>Model construction</li>
<li>Performance evaluation</li>
<li>Model improvement</li>
</ul>

Data Source: "https://www.kaggle.com/datasets/psycon/daily-gold-price-historical-data"

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as numpy
import pandas as pandas
import matplotlib.pyplot as pyplot
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn import metrics
from scipy.optimize import curve_fit

# Data Exploration

In [ ]:
# create a dataset from source csv file
RawData = pandas.read_csv('/kaggle/input/daily-gold-price-historical-data/gold.csv')
# view some rows of the dataset
RawData.head()

In [ ]:
# get information about the dataset
RawData.info()

In [ ]:
# calculate an atomic value that represents the daily gold price
Dataset = RawData
Dataset['Value'] = (Dataset['High'] + Dataset['Low']) / 2
Dataset = Dataset[['Date', 'Value']]
# plot line chart to visualize data
pyplot.plot(Dataset['Date'], Dataset['Value'])
pyplot.xlabel("Date")
pyplot.ylabel("Value")
pyplot.title("Gold Price ($)")
pyplot.show()

# Linear Regression

It is obvious that time series forecasting methods should be used in this case. As it can be seen on the line chart above, the gold prices have positive trend since 2000. Hence regression model is a good method to make predictions

In [ ]:
# transform data set date values to time series indices
def TransformDateToIndex():
    counter = 1
    for x in Dataset['Date']:
        Dataset['Date'].iat[counter - 1] = counter
        counter = counter + 1
TransformDateToIndex()

# reshape dataset axes in order to make regression properly
X_values = Dataset['Date'].values.reshape(-1, 1)
Y_values = Dataset['Value'].values.reshape(-1, 1)

# split the dataset for training and testing
X_train, X_test, Y_train, Y_test = train_test_split(X_values, Y_values, test_size = 0.2)

# create linear regression instance
regression = LinearRegression()

# fit linear regression line using training data
regression.fit(X_train, Y_train)

# make predictions from independent test variable
Y_pred = regression.predict(X_test)

# visualize raw data and regression model on same graph
pyplot.plot(Dataset['Date'], Dataset['Value'])
pyplot.plot(X_test, Y_pred)
pyplot.title("Linear Regression Model")
pyplot.xlabel("Date")
pyplot.ylabel("Value")
pyplot.show()

# show intercept value of regression line
print("intercept:", regression.intercept_)

# show slope value of regression line
print("slope:", regression.coef_)
print("\n")

# show forecasting model metrics for performance evaluation
print("Mean Absolute Error: ", metrics.mean_absolute_error(Y_test, Y_pred))
print("R-Squared: ", metrics.r2_score(Y_test, Y_pred))
print("\n")

# show actual test data and predicted data
ResultsDataFrame = pandas.DataFrame({'Actual Data' : Y_test.squeeze(), 'Predicted Data' : Y_pred.squeeze()})
print(ResultsDataFrame)

R Squared value of the linear regression model shows that the model can explain approximately %80 of the variation in gold prices. 

# Polynomial Regression

As it mentioned before, the gold price data is trending. Moreover it show some curves throughout the time axis. İn order to improve the efficiency of the forecasting model, curvilinear effects should be taken into account in the model.

In [ ]:
# define mapping function
def Mapping(x, a, b, c, d, e):
    return a + b*x + c*x**2 + d*x**3 + e*x**4
    
# call curve_fit function in order to find optimal values of model parameters
OptimizedParameters, _ = curve_fit(Mapping, Dataset['Date'], Dataset['Value'])

a, b, c, d, e= OptimizedParameters

Predictions = Mapping(Dataset['Date'], a, b, c, d, e)

# visualize raw data and regression model on same graph
pyplot.plot(Dataset['Date'], Dataset['Value'])
pyplot.plot(Dataset['Date'], Predictions)
pyplot.title("Polynomial Regression Model")
pyplot.xlabel("Date")
pyplot.ylabel("Value")
pyplot.show()

# show model metrics
print("Mean Absolute Error: ", metrics.mean_absolute_error(Dataset['Value'], Predictions))
print("R-Squared: ", metrics.r2_score(Dataset['Value'], Predictions))
print("\n")

# show next prediction
print("Next Prediction: ", Mapping(len(Dataset), a, b, c, d, e))
print("\n")

# show actual test data and predicted data
TestResultsDataFrame = pandas.DataFrame({'Actual Data' : Dataset['Value'], 'Predicted Data' : Predictions})
print(TestResultsDataFrame)


R Squared value of the polynomial regression model show that the model can explain approximately %90 of the variation in gold prices. Also the error measurement is reduced. That refers to an improvement in the predictive model.